# Multi-Site Ecological Monitoring Dashboard

This notebook demonstrates how to use **Kinship Earth** to build a comparative ecological monitoring view across multiple sites. We query five diverse ecosystems around the world, compare biodiversity and climate metrics, and export the results for GIS analysis.

This is a common workflow for researchers managing multi-site studies, conservation planning, or ecological assessments.

**What you will do:**
1. Define a set of monitoring locations
2. Query each location for species observations and environmental context
3. Build a comparison DataFrame
4. Visualize biodiversity across sites
5. Export results as GeoJSON files for QGIS

**Prerequisites:** A running Kinship Earth MCP server and the dependencies from `getting-started.ipynb`.

In [ ]:
%pip install mcp httpx pandas matplotlib

## 1. Define Monitoring Locations

We will query five ecologically significant sites spanning marine, freshwater, and terrestrial biomes. Each site is defined by a name, coordinates, and a search radius.

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from mcp import ClientSession
from mcp.client.stdio import StdioServerParameters, stdio_client

# --- Server connection ---
# Adjust the path to point to your Kinship Earth installation
server_params = StdioServerParameters(
    command="uv",
    args=["run", "--directory", "/path/to/kinship-earth-mcp/servers/orchestrator",
          "python", "-m", "kinship_orchestrator.server"],
    env={},
)

# --- Monitoring sites ---
# Each site: name, latitude, longitude, search radius (km), biome description
SITES = [
    {
        "name": "Monterey Bay",
        "lat": 36.6,
        "lon": -121.9,
        "radius_km": 50,
        "biome": "Temperate marine (California Current)",
    },
    {
        "name": "Yellowstone",
        "lat": 44.6,
        "lon": -110.5,
        "radius_km": 60,
        "biome": "Montane / geothermal terrestrial",
    },
    {
        "name": "Everglades",
        "lat": 25.3,
        "lon": -80.9,
        "radius_km": 50,
        "biome": "Subtropical wetland / mangrove",
    },
    {
        "name": "Great Barrier Reef",
        "lat": -18.3,
        "lon": 147.7,
        "radius_km": 80,
        "biome": "Tropical coral reef",
    },
    {
        "name": "Amazon (Manaus)",
        "lat": -3.1,
        "lon": -60.0,
        "radius_km": 80,
        "biome": "Tropical rainforest / freshwater",
    },
]

print(f"Monitoring {len(SITES)} sites:")
for site in SITES:
    print(f"  - {site['name']} ({site['biome']})")

## 2. Query Each Location

We iterate over the monitoring sites and call `ecology_search` for each one. The results include species occurrences, nearby NEON sites (for US locations), and climate context.

We also call `ecology_get_environmental_context` to get detailed climate data for the past week.

In [ ]:
from datetime import date, timedelta

# Date range: query environmental context for the past week
today = date.today().isoformat()
one_week_ago = (date.today() - timedelta(days=7)).isoformat()

# Store results for each site
site_results = []

async with stdio_client(server_params) as (read, write):
    async with ClientSession(read, write) as session:
        await session.initialize()

        for site in SITES:
            print(f"Querying {site['name']}...")

            # --- Species search ---
            search_result = await session.call_tool(
                "ecology_search",
                arguments={
                    "lat": site["lat"],
                    "lon": site["lon"],
                    "radius_km": site["radius_km"],
                    "limit": 100,
                    "include_climate": True,
                    "output_format": "geojson",
                },
            )
            search_data = json.loads(search_result.content[0].text)

            # --- Environmental context ---
            env_result = await session.call_tool(
                "ecology_get_environmental_context",
                arguments={
                    "lat": site["lat"],
                    "lon": site["lon"],
                    "date": today,
                    "days_before": 7,
                },
            )
            env_data = json.loads(env_result.content[0].text)

            site_results.append({
                "site": site,
                "search": search_data,
                "environment": env_data,
            })

            species_count = len(search_data.get("species_occurrences", []))
            neon_count = len(search_data.get("neon_sites", []))
            print(f"  -> {species_count} observations, {neon_count} NEON sites")

print(f"\nAll {len(SITES)} sites queried.")

## 3. Build a Comparison DataFrame

Let's extract key metrics from each site's results into a single DataFrame for comparison: number of species, number of observations, unique taxa, NEON site count, and a climate summary.

In [ ]:
comparison_rows = []

for entry in site_results:
    site = entry["site"]
    search = entry["search"]
    env = entry["environment"]

    occurrences = search.get("species_occurrences", [])
    occ_df = pd.DataFrame(occurrences) if occurrences else pd.DataFrame()

    # Count unique species (by scientific_name if available)
    name_col = "scientific_name" if "scientific_name" in occ_df.columns else "scientificName"
    unique_species = occ_df[name_col].nunique() if name_col in occ_df.columns else 0

    # Extract mean temperature from climate data (if available)
    climate_daily = env.get("climate", {}).get("daily", [])
    if climate_daily:
        temps = []
        for day in climate_daily:
            # Try common temperature field names
            for key in ["temperature_2m_mean", "mean_temp", "temperature_mean"]:
                if key in day and day[key] is not None:
                    temps.append(day[key])
                    break
        avg_temp = round(sum(temps) / len(temps), 1) if temps else None
    else:
        avg_temp = None

    comparison_rows.append({
        "Site": site["name"],
        "Biome": site["biome"],
        "Latitude": site["lat"],
        "Longitude": site["lon"],
        "Total Observations": len(occurrences),
        "Unique Species": unique_species,
        "NEON Sites Nearby": len(search.get("neon_sites", [])),
        "Avg Temp (C)": avg_temp,
    })

comparison_df = pd.DataFrame(comparison_rows)
comparison_df

## 4. Visualize Biodiversity Across Sites

A bar chart comparing species richness (unique species count) and total observations across our five monitoring sites.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = ["#2E86AB", "#A23B72", "#F18F01", "#C73E1D", "#3B1F2B"]

# --- Species richness ---
axes[0].barh(
    comparison_df["Site"],
    comparison_df["Unique Species"],
    color=colors,
)
axes[0].set_xlabel("Unique Species")
axes[0].set_title("Species Richness by Site")
axes[0].invert_yaxis()

# Add value labels
for i, v in enumerate(comparison_df["Unique Species"]):
    axes[0].text(v + 0.5, i, str(v), va="center", fontsize=10)

# --- Total observations ---
axes[1].barh(
    comparison_df["Site"],
    comparison_df["Total Observations"],
    color=colors,
)
axes[1].set_xlabel("Total Observations")
axes[1].set_title("Observation Count by Site")
axes[1].invert_yaxis()

for i, v in enumerate(comparison_df["Total Observations"]):
    axes[1].text(v + 0.5, i, str(v), va="center", fontsize=10)

plt.suptitle("Multi-Site Ecological Comparison — Kinship Earth", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## 5. Temperature Comparison Across Sites

If climate data was returned, let's compare the average temperature across all five sites.

In [ ]:
# Filter to sites where we have temperature data
temp_df = comparison_df.dropna(subset=["Avg Temp (C)"])

if not temp_df.empty:
    fig, ax = plt.subplots(figsize=(10, 4))

    bars = ax.bar(
        temp_df["Site"],
        temp_df["Avg Temp (C)"],
        color=colors[:len(temp_df)],
        edgecolor="white",
        linewidth=0.8,
    )

    # Add value labels on bars
    for bar, val in zip(bars, temp_df["Avg Temp (C)"]):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.3,
            f"{val} C",
            ha="center",
            fontsize=10,
        )

    ax.set_ylabel("Average Temperature (C)")
    ax.set_title("7-Day Average Temperature by Site")
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("No temperature data available for comparison.")

## 6. Export Results as GeoJSON for QGIS

We save each site's species observations as a separate GeoJSON file. These can be loaded directly into QGIS as vector layers for spatial analysis, overlay with other datasets, and map production.

**QGIS import instructions:**
1. Open QGIS
2. Go to Layer > Add Layer > Add Vector Layer
3. Select the `.geojson` file
4. The observations will appear as point features on the map

In [ ]:
output_dir = Path("geojson_export")
output_dir.mkdir(exist_ok=True)

for entry in site_results:
    site = entry["site"]
    search = entry["search"]

    # Use the GeoJSON output if available, otherwise build it from occurrences
    geojson = search.get("species_occurrences_geojson")

    if geojson is None:
        # Manually construct GeoJSON from flat occurrence records
        occurrences = search.get("species_occurrences", [])
        features = []
        for obs in occurrences:
            lat = obs.get("lat") or obs.get("decimalLatitude")
            lon = obs.get("lon") or obs.get("lng") or obs.get("decimalLongitude")
            if lat is not None and lon is not None:
                # GeoJSON coordinates are [lon, lat]
                features.append({
                    "type": "Feature",
                    "geometry": {
                        "type": "Point",
                        "coordinates": [lon, lat],
                    },
                    "properties": {
                        k: v for k, v in obs.items()
                        if k not in ("lat", "lon", "lng", "decimalLatitude", "decimalLongitude")
                    },
                })
        geojson = {"type": "FeatureCollection", "features": features}

    # Write to file
    filename = site["name"].lower().replace(" ", "_").replace("(", "").replace(")", "") + ".geojson"
    filepath = output_dir / filename
    filepath.write_text(json.dumps(geojson, indent=2))
    print(f"Saved {filepath} ({len(geojson.get('features', []))} features)")

# Also save the comparison table as CSV
csv_path = output_dir / "site_comparison.csv"
comparison_df.to_csv(csv_path, index=False)
print(f"\nComparison table saved to {csv_path}")
print(f"\nAll files saved to {output_dir.resolve()}/")

## Next Steps

With the GeoJSON files exported, you can:

- **Overlay in QGIS** with habitat maps, protected area boundaries, or satellite imagery
- **Run spatial analyses** — buffer zones, species range intersections, connectivity modeling
- **Schedule monitoring** — run this notebook periodically to track how species observations and climate metrics change over time
- **Add more sites** — simply add entries to the `SITES` list above
- **Filter by species** — add a `scientificname` parameter to `ecology_search` to track a specific taxon across sites

For more information:
- [Kinship Earth on GitHub](https://github.com/christineyws-beep/kinship-earth-mcp)
- [Getting Started notebook](./getting-started.ipynb) — single-site queries and visualization basics